# 03 Terrain and capacity

Tests whether the master plan's stated comfortable carrying capacity (CCC) can be reproduced from its own stated per-lift inputs (claim C025), using the formula the appendix itself describes (claim C024). This notebook does not attempt the LiDAR slope-by-ability-class analysis in the step's original question; see Open issues in steps/03 for why.

In [ ]:
research_dir = "research"
processed_dir = "data/processed"
rounding_tolerance = 2

## Per-lift table (transcribed from Appendix A)

The rows below are transcribed from data/raw/S011_rmr_appendix_a_ccc_tables.pdf, Tables 1-1 to 1-4, via pdfplumber's table extraction (pypdf's plain-text extraction jumbled these tables' row order, so pdfplumber's structured extraction was used instead and checked by hand against the same tables' raw text). Kept as a literal transcription rather than re-parsing the PDF here, same reasoning as notebook 02: one place does the parsing, everything else reads the result.

In [ ]:
LIFT_ROWS = """
phase,map_ref,slope_length_m,vert_rise_m,hourly_capacity,oper_hours,access_role_pct,misload_pct,stated_adjusted_hourly,stated_vtm_day_000,weighted_vertical_demand_m,stated_ccc
Phase 1 (Existing),Lil' Bit,103,15.45,1500,7,75,2,345,1000,1000,37
Phase 1 (Existing),Turtle Creek Carpet,176,30.5,1800,7,0,2,1764,1000,1000,377
Phase 1 (Existing),2,1011,280,2185,7,85,10,109,0,1583,135
Phase 1 (Existing),5,2391,900,2357,7,75,10,353,879,3394,656
Phase 1 (Existing),12,1902,633,2200,6.5,5,5,1980,6829,3379,2411
Phase 1 (Existing),14,1880,530,1800,6,0,5,1710,6473,5857,928
Phase 2,Lil' Bit,103,15.45,1500,7,75,2,345,1000,1000,37
Phase 2,Alpine Carpet,250,50,1800,7,85,2,234,1000,1000,82
Phase 2,2,1011,280,2800,7,85,10,140,0,1583,173
Phase 2,5,2391,900,2800,7,75,10,420,879,3394,780
Phase 2,12,1902,633,2600,6.5,5,5,2340,6829,3379,2849
Phase 2,14,1880,530,2600,6,0,5,2470,6473,5857,1341
Phase 2,3,1006,280,2400,7,50,10,960,6268,4350,433
Phase 2,6,1983,658,2400,7,30,10,1440,8900,7150,928
Phase 2,11,438,125,1800,7,5,10,1530,1342,3500,383
Phase 2,13,1216,412,1800,6.5,0,10,1620,1715,9625,451
Phase 2,15,2111,885,2400,6,0,10,2160,9232,8625,1330
Phase 2,18,1235,444,2400,6.5,0,10,2160,7322,8400,742
Phase 3,Lil' Bit,103,15.45,1500,7,75,2,345,1000,1000,37
Phase 3,Alpine Carpet,250,50,1800,7,85,2,234,1000,1000,82
Phase 3,2,1011,280,2800,7,85,10,140,0,1583,173
Phase 3,5,2391,900,2800,7,75,10,420,879,3394,780
Phase 3,12,1902,633,2600,6.5,5,5,2340,6829,3379,2849
Phase 3,14,1880,530,2600,6,0,5,2470,6473,5857,1341
Phase 3,3,1006,280,2400,7,50,10,960,6268,4350,433
Phase 3,6,1983,658,2400,7,30,10,1440,8900,7150,928
Phase 3,11,438,125,1800,7,5,10,1530,1342,3500,383
Phase 3,13,1216,412,1800,6.5,0,10,1620,1715,9625,451
Phase 3,15,2111,885,2400,6,0,10,2160,9232,8625,1330
Phase 3,18,1235,444,2400,6.5,0,10,2160,7322,8400,742
Phase 3,19,1193,397,1500,6,0,5,1425,3392,7934,428
Phase 3,21,513,63,1800,6.5,100,0,0,0,0,0
Phase 3,22,1225,500,2400,6.8,15,5,1920,6527,4423,1476
Phase 3,23,189,21,1000,6,0,10,900,116,1000,113
Phase 3,24,120,9,500,6,0,5,475,26,1000,26
Phase 3,25,1093,451,2400,6,100,0,0,0,0,0
Buildout,Lil' Bit,103,15.45,1500,7,75,2,345,1000,1000,37
Buildout,Alpine Carpet,250,50,1800,7,85,2,234,1000,1000,82
Buildout,2,1011,280,2800,7,85,10,140,0,1583,173
Buildout,5,2391,900,2800,7,75,10,420,879,3394,780
Buildout,12,1902,633,2600,6.5,5,5,2340,6829,3379,2849
Buildout,14,1880,530,2600,6,0,5,2470,6473,5857,1341
Buildout,3,1006,280,2400,7,50,10,960,6268,4350,433
Buildout,6,1983,658,2400,7,30,10,1440,8900,7150,928
Buildout,11,438,125,1800,7,5,10,1530,1342,3500,383
Buildout,13,1216,412,1800,6.5,0,10,1620,1715,9625,451
Buildout,15,2111,885,2400,6,0,10,2160,9232,8625,1330
Buildout,18,1235,444,2400,6.5,0,10,2160,7322,8400,742
Buildout,19,1193,397,1500,6,0,5,1425,3392,7934,428
Buildout,21,513,63,1800,6.5,100,0,0,0,0,0
Buildout,22,1225,500,2400,6.8,15,5,1920,6527,4423,1476
Buildout,23,189,21,1000,6,0,10,900,116,1000,113
Buildout,24,120,9,500,6,0,5,475,26,1000,26
Buildout,25,1093,451,2400,6,100,0,0,0,0,0
Buildout,1,1198,382,3000,7,75,10,450,1203,2500,481
Buildout,4,1160,399,2800,7,85,10,140,391,5000,78
Buildout,7,1358,421,2800,7,5,5,2520,7420,5166,1438
Buildout,8,976,164,1800,7,100,0,0,0,0,0
Buildout,9,701,94,1800,6.5,0,15,1530,931,2500,374
Buildout,10,1559,414,2400,6.8,10,5,2040,5746,4560,1259
Buildout,16,1313,395,1800,6.5,50,10,720,1847,8500,217
Buildout,17,1993,633,2400,6.5,5,5,2160,8885,5441,1633
Buildout,20,1746,579,2400,6.25,0,5,2280,8253,7403,1115
"""

## Load the ledger

Confirms C024 and C025, the claims behind the formula and the phase totals this notebook tries to reproduce, still point to the same source before using them.

In [ ]:
import io
import sys

import pandas as pd

sys.path.insert(0, "src")
from resort.ledger import read_ledger

ledger = read_ledger(
    claims_path=f"{research_dir}/claims.csv",
    sources_path=f"{research_dir}/sources.csv",
)
assert ledger["claims"]["C024"]["source_id"] == "S011"
assert ledger["claims"]["C025"]["source_id"] == "S011"

lifts = pd.read_csv(io.StringIO(LIFT_ROWS))
print(f"{len(lifts)} (phase, lift) rows across {lifts['phase'].nunique()} phases")

## Reproduce Adjusted Hourly Capacity and CCC

C024 gives the top-level formula, CCC = Vertical Rise x Hourly Capacity x Operating Hours x Loading Efficiency / Weighted Vertical Demand, but the appendix text does not spell out Loading Efficiency as its own number. Testing arithmetic against the table's own Adjusted Hourly Capacity column shows it equals Hourly Capacity x (1 - Up-Mtn Access Role% - Misload Lift Stop%); CCC then follows from Adjusted Hourly Capacity x Vert Rise x Oper Hours / Weighted Vertical Demand. Neither equation is a sentence written in the source, so this is this notebook's own reverse-engineering of the tables (a judgment call, not a claim about what the text says), checked below against every row and phase total the appendix actually states.

In [ ]:
lifts["computed_adjusted_hourly"] = lifts["hourly_capacity"] * (
    1 - lifts["access_role_pct"] / 100 - lifts["misload_pct"] / 100
)
lifts["computed_ccc"] = (
    lifts["computed_adjusted_hourly"] * lifts["vert_rise_m"] * lifts["oper_hours"]
    / lifts["weighted_vertical_demand_m"].replace(0, pd.NA)
).fillna(0)
lifts["ccc_diff"] = (lifts["computed_ccc"] - lifts["stated_ccc"]).round(1)
lifts[["phase", "map_ref", "stated_ccc", "computed_ccc", "ccc_diff"]]

## Compare phase totals against step 02's output

Reads data/processed/02_ccc_by_phase.csv (written by notebook 02 from the same claim, C025) rather than retyping the four phase totals a second time, so the two notebooks cannot silently drift apart on the same number.

In [ ]:
ccc_by_phase = pd.read_csv(f"{processed_dir}/02_ccc_by_phase.csv")
computed_by_phase = (
    lifts.groupby("phase", sort=False)["computed_ccc"].sum().round().astype(int)
)
phase_order = ["Phase 1 (Existing)", "Phase 2", "Phase 3", "Buildout"]
comparison = ccc_by_phase.set_index("phase").loc[phase_order]
comparison["computed_ccc_skiers"] = computed_by_phase.loc[phase_order].values
comparison["diff"] = comparison["computed_ccc_skiers"] - comparison["ccc_skiers"]
comparison

## Write outputs

The per-lift reproduction and the phase-level comparison, so step 08's synthesis can cite how well the chain's first link actually reproduces, not just its final numbers.

In [ ]:
import os

os.makedirs(processed_dir, exist_ok=True)
lifts.to_csv(f"{processed_dir}/03_lift_ccc_reproduction.csv", index=False, encoding="utf-8")
comparison.to_csv(f"{processed_dir}/03_ccc_phase_reproduction_summary.csv", encoding="utf-8")
print("wrote 03_lift_ccc_reproduction.csv, 03_ccc_phase_reproduction_summary.csv")

## Checks

Every phase total, and every individual lift's CCC, should match the appendix's own stated figure within a small rounding tolerance. A mismatch bigger than that would mean either this notebook's transcription has an error or the reverse-engineered formula is wrong, and either way it should fail loudly rather than report a reproduction that did not actually happen.

In [ ]:
max_lift_diff = lifts["ccc_diff"].abs().max()
max_phase_diff = comparison["diff"].abs().max()
assert max_lift_diff <= rounding_tolerance, f"largest per-lift CCC diff was {max_lift_diff}"
assert max_phase_diff <= rounding_tolerance, f"largest phase-total CCC diff was {max_phase_diff}"
print(f"checks passed: max per-lift diff {max_lift_diff}, max phase-total diff {max_phase_diff}")

## Versions

In [ ]:
import importlib.metadata
import sys

print("python", sys.version)
for pkg in ["pandas"]:
    print(pkg, importlib.metadata.version(pkg))